# Tiingo end-of-day prices

Tiingo publishes daily OHLCV for US stocks, ETFs and mutual funds, back to each
instrument's first trading day -- 1980 for `AAPL`. Alongside the prices as
traded it publishes the *adjusted* series, every bar restated for the splits and
dividends that came after it.

What is surprising is what is absent: there is no catalog to search. Every other
source on this server hands you a list of series to pick through; Tiingo expects
you to arrive already knowing the symbol. Discovery here is metadata rather than
search -- `tiingo_series_info` reports the coverage window, and asking it first
is what keeps you from requesting data that was never published.

**Requires the MCP server running** (`python mcp_server/server.py`) and
`TIINGO_API_KEY` in `../navi/.env`.

In [ ]:
%reload_ext autoreload
%autoreload 2

import sys
sys.path.append('../')

from matplotlib import pyplot
from lib import config

from utils import (
    list_mcp_tools,
    show_tool_schema,
    get_series_info,
    get_price_series,
    plot_price_series,
)

pyplot.style.use(config.glyfish_style)

## 1. Discovery

Two tools, and they split the work cleanly: one says what a symbol *is* and how
far back it goes, the other returns bars. `list_mcp_tools()` with no argument
lists every tool on the server; the prefix narrows it to this source.

In [ ]:
_ = await list_mcp_tools('tiingo_')

### The schema of the tool we fetch with

`show_tool_schema` prints the **output** schema as well as the input one, and
here the output half is the one a caller needs. It fixes the request shape --
`start_date`/`end_date` as `YYYY-MM-DD`, and omitting them returns only the
latest day -- and then names every field that comes back.

Those names are meida's, not Tiingo's. The wire format is camelCase (`adjClose`,
`divCash`, `splitFactor`) and the response models translate it, so a row arrives
as `adj_close` / `div_cash` / `split_factor` with an ISO `date` string, matching
every other source on the server.

`count` is worth reading twice: it is how many rows *Tiingo sent*, not how many
were mapped, so `count != len(prices)` means rows were dropped in translation
rather than never sent.

In [ ]:
_ = await show_tool_schema('tiingo_price_series')

## 2. Finding a series

With no search tool, the identifier has to come from you -- an exchange symbol,
`AAPL`. What the server can do is confirm the symbol resolves and report what it
covers, which is the closest thing to discovery Tiingo offers and the more
useful half regardless: a symbol you guessed wrong fails loudly here, instead of
quietly returning nothing later.

`start_date` and `end_date` are the coverage window, and they are the reason to
make this call before fetching. A request outside the window is not an error --
Tiingo answers with an empty list, which is indistinguishable from a bad symbol
or a wrong date format unless you already know when the history begins.

In [ ]:
info = await get_series_info('AAPL')

In [ ]:
empty = await get_price_series('AAPL', start_date='1970-01-01', end_date='1979-12-31')
print(f"a decade before {info['start_date']}: {len(empty['prices'])} rows, and no error")

## 3. Fetch and plot

Now a range inside the window: sixteen years of `AAPL`, spanning two stock
splits and the dividend it reinstated in 2012. The corporate actions ride along
in the rows themselves -- `split_factor` is `1.0` on an ordinary day and the
split ratio on the day one takes effect -- so they can be pulled straight out of
the series before plotting it.

In [ ]:
series = await get_price_series('AAPL', start_date='2010-01-01', end_date=info['end_date'], show=3)

In [ ]:
for row in series['prices']:
    if row['split_factor'] != 1.0:
        print(f"{row['date']}  split_factor={row['split_factor']}  "
              f"close={row['close']:>8.2f}  adj_close={row['adj_close']:>8.2f}")

Both closes go on one axis, because the gap between them is the whole point.
`close` is the price as traded, so it falls by a factor of seven overnight in
June 2014 and by four again in August 2020 -- moves that say nothing about what
the company is worth, only that the shares were cut up. `adj_close` restates the
entire history for those splits and for reinvested dividends, so its slope is
the return an investor actually earned.

That is why returns are computed from `adj_close` and never from `close`: a
return series built on the raw closes would read those two split days as
catastrophic losses.

In [ ]:
plot_price_series(series, fields=('close', 'adj_close'), info=info)